# QuechuaTok: Morphological Tokenization Benchmark for Southern Quechua
**Author:** Maky (Maria Contreras) | [@macmaky](https://www.kaggle.com/macmaky)
Paper: https://arxiv.org/abs/2606.23943

Comparative benchmark of BPE, Unigram LM and WordPiece tokenizers 
for Southern Quechua (low-resource agglutinative language). 
Uses Llamacha/monolingual-quechua-iic corpus. 
Evaluates fertility rate, OOV% and morphological boundary accuracy.
Part of the QuechuaTok research project.

**Dataset:** `Llamacha/monolingual-quechua-iic` (Southern Quechua, ~Wikipedia + OSCAR)

---

In [ ]:
# HF_TOKEN optional — only needed for faster HuggingFace downloads
# Uncomment and set your token if needed:
# import os; os.environ["HF_TOKEN"] = "your_token_here"
print("Setup ready ✓")

In [ ]:
# Install dependencies
!pip install datasets sentencepiece tokenizers unicodedata2 -q

In [ ]:
from datasets import load_dataset
import sentencepiece as spm
from tokenizers import BertWordPieceTokenizer
import re, json, unicodedata
from pathlib import Path
from collections import Counter
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

Path('data/processed').mkdir(parents=True, exist_ok=True)
Path('models').mkdir(parents=True, exist_ok=True)
print('Setup complete ✓')

## 1. Corpus Download

In [ ]:
ds = load_dataset('Llamacha/monolingual-quechua-iic', split='train')
print(f'Filas: {len(ds):,}')
print(f'Columnas: {ds.column_names}')
print(f'Ejemplo: {ds[0]}')

texts = [item['text'] for item in ds if item.get('text','').strip()]
print(f'\nNon-empty texts: {len(texts):,}')

## 2. Preprocessing

In [ ]:
def normalize(text):
    text = unicodedata.normalize('NFC', text).lower()
    text = re.sub(r'https?://\S+|@\w+|#\w+', ' ', text)
    text = re.sub(r'[^\w\s\']', ' ', text)
    return re.sub(r'\s+', ' ', text).strip()

QUECHUA_SUFFIXES = ['kuna','pi','manta','wan','paq','ta','qa','pas',
                    'ni','nki','n','nchik','nku','mi','si','cha',
                    'rqa','sqa','nqa','y','q','na','spa']

def is_valid(text, min_tok=3):
    tokens = text.split()
    if len(tokens) < min_tok: return False
    hits = sum(1 for t in tokens if any(t.endswith(s) for s in QUECHUA_SUFFIXES))
    return (hits / len(tokens)) >= 0.1

clean = [normalize(t) for t in texts]
clean = [t for t in clean if t and is_valid(t)]
clean = list(dict.fromkeys(clean))  # deduplicate

print(f'Original lines:  {len(texts):,}')
print(f'Clean lines:     {len(clean):,} ({len(clean)/len(texts)*100:.1f}% retenido)')

# Split train/test
test_sents = clean[-1000:]
train_sents = clean[:-1000]

with open('data/processed/quechua_train.txt', 'w') as f:
    f.write('\n'.join(train_sents))

print(f'Train: {len(train_sents):,} | Test: {len(test_sents):,}')

## 3. Tokenizer Training

In [ ]:
import os
from pathlib import Path

VOCAB = 4000
TRAIN = 'data/processed/quechua_train.txt'
TRAIN_SMALL = 'data/processed/quechua_train_small.txt'

with open(TRAIN) as f:
    lines = f.readlines()

sample = lines[:50000]
with open(TRAIN_SMALL, 'w') as f:
    f.writelines(sample)

print(f'Sample: {len(sample):,} líneas')

spm.SentencePieceTrainer.train(
    input=TRAIN_SMALL,
    model_prefix='models/bpe',
    vocab_size=VOCAB,
    model_type='bpe',
    character_coverage=0.9999,
    byte_fallback=True,
    user_defined_symbols=["'"]
)
bpe = spm.SentencePieceProcessor()
bpe.load('models/bpe.model')
print('BPE ✓')

In [ ]:
spm.SentencePieceTrainer.train(
    input=TRAIN_SMALL,
    model_prefix='models/unigram',
    vocab_size=VOCAB,
    model_type='unigram',
    character_coverage=0.9999,
    byte_fallback=True,
    user_defined_symbols=["'"]
)
uni = spm.SentencePieceProcessor()
uni.load('models/unigram.model')
print('Unigram ✓')

In [ ]:
wp = BertWordPieceTokenizer(lowercase=True, strip_accents=False)
wp.train(files=[TRAIN_SMALL], vocab_size=VOCAB, min_frequency=2)
Path('models/wordpiece').mkdir(exist_ok=True)
wp.save_model('models/wordpiece')
print('WordPiece ✓')

## 4. Comparative Evaluation

In [ ]:
def fertility(tokenizer, sentences, typ='spm'):
    tots, words = 0, 0
    for s in sentences:
        words += len(s.split())
        tots += len(tokenizer.encode(s, out_type=str) if typ=='spm'
                    else tokenizer.encode(s).tokens)
    return round(tots/max(words,1), 4)

def oov_rate(tokenizer, sentences, typ='spm'):
    tots, oovs = 0, 0
    for s in sentences:
        if typ=='spm':
            ids = tokenizer.encode(s); tots+=len(ids); oovs+=ids.count(1)
        else:
            toks = tokenizer.encode(s).tokens
            tots+=len(toks); oovs+=toks.count('[UNK]')
    return round(oovs/max(tots,1)*100, 4)

results = {
    'BPE':      {'fertility': fertility(bpe, test_sents),
                 'oov_%':    oov_rate(bpe, test_sents),
                 'vocab':    bpe.get_piece_size()},
    'Unigram':  {'fertility': fertility(uni, test_sents),
                 'oov_%':    oov_rate(uni, test_sents),
                 'vocab':    uni.get_piece_size()},
    'WordPiece':{'fertility': fertility(wp, test_sents, 'wp'),
                 'oov_%':    oov_rate(wp, test_sents, 'wp'),
                 'vocab':    wp.get_vocab_size()},
}

df = pd.DataFrame(results).T
print(df.to_string())

In [ ]:
# Comparative visualization
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
fig.suptitle('Comparativa de tokenizadores — Quechua del Sur', fontsize=13)

names = list(results.keys())
colors = ['#E24B4A', '#1D9E75', '#378ADD', '#BA7517']

ax1.bar(names, [results[n]['fertility'] for n in names], color=colors)
ax1.set_title('Fertility rate (tokens/palabra)\n← lower is better')
ax1.set_ylim(0, max(results[n]['fertility'] for n in names) * 1.3)

ax2.bar(names, [results[n]['oov_%'] for n in names], color=colors)
ax2.set_title('OOV rate (%)\n← lower is better')
ax2.yaxis.set_major_formatter(ticker.FormatStrFormatter('%.1f%%'))

plt.tight_layout()
plt.savefig('data/processed/tokenizer_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved ✓')

## 5. Qualitative Analysis

In [ ]:
# Experimento 2: vocab_size=8000
VOCAB2 = 8000

spm.SentencePieceTrainer.train(
    input=TRAIN_SMALL,
    model_prefix='models/bpe_8k',
    vocab_size=VOCAB2,
    model_type='bpe',
    character_coverage=0.9999,
    byte_fallback=True,
    user_defined_symbols=["'"]
)
bpe_8k = spm.SentencePieceProcessor()
bpe_8k.load('models/bpe_8k.model')

spm.SentencePieceTrainer.train(
    input=TRAIN_SMALL,
    model_prefix='models/unigram_8k',
    vocab_size=VOCAB2,
    model_type='unigram',
    character_coverage=0.9999,
    byte_fallback=True,
    user_defined_symbols=["'"]
)
uni_8k = spm.SentencePieceProcessor()
uni_8k.load('models/unigram_8k.model')

print('Modelos 8k entrenados ✓')

In [ ]:
examples2 = [
    ('rimankichikmi',     'rima | nkichik | mi'),
    ('wasiykikunapiqa',   'wasi | yki | kuna | pi | qa'),
    ('purisqanchikmanta', 'puri | sqa | nchik | manta'),
    ('munakuwarqanki',    'muna | ku | wa | rqa | nki'),
]

print(f"{'Palabra':<25} {'Gold':<28} {'BPE 8k':<35} {'Unigram 8k'}")
print('─' * 110)
for word, gold in examples2:
    bpe_seg = ' | '.join(bpe_8k.encode(word, out_type=str))
    uni_seg = ' | '.join(uni_8k.encode(word, out_type=str))
    print(f"{word:<25} {gold:<28} {bpe_seg:<35} {uni_seg}")

In [ ]:
# PRPE morfológico quechua
from collections import Counter

# Quechua morphological suffixes organized by category
SUFFIXES = sorted([
    # Persona verbal
    'ni','nki','n','nchik','nkichik','nku',
    # TAM
    'rqa','sqa','nqa','rqan','sqan',
    # Nominales
    'kuna','pi','manta','wan','paq','ta','qa','pas',
    # Evidenciales
    'mi','si','cha',
    # Derivativos
    'stin','spa','na',
], key=len, reverse=True)

def segment_prpe(word):
    if len(word) <= 3:
        return [word]
    segments = []
    remaining = word
    while len(remaining) > 3:
        found = False
        for suf in SUFFIXES:
            if remaining.endswith(suf) and len(remaining)-len(suf) >= 3:
                segments.insert(0, suf)
                remaining = remaining[:-len(suf)]
                found = True
                break
        if not found:
            break
    segments.insert(0, remaining)
    return segments

# Fertility rate del PRPE
total_segs, total_words = 0, 0
for s in test_sents:
    words = s.split()
    total_words += len(words)
    for w in words:
        total_segs += len(segment_prpe(w))

prpe_fertility = round(total_segs / max(total_words, 1), 4)
print(f'PRPE fertility rate: {prpe_fertility}')

# Tabla final completa
print(f"\n{'Modelo':<20} {'Fertility':<12} {'OOV%'}")
print('─' * 44)
for name, tok, typ in [('BPE 4k', bpe, 'spm'), 
                        ('BPE 8k', bpe_8k, 'spm'),
                        ('Unigram 4k', uni, 'spm'), 
                        ('Unigram 8k', uni_8k, 'spm')]:
    print(f"{name:<20} {fertility(tok, test_sents, typ):<12} {oov_rate(tok, test_sents, typ)}")
print(f"{'PRPE morfológico':<20} {prpe_fertility:<12} {'0.0 (by design)'}")

In [ ]:
import matplotlib.pyplot as plt

modelos = ['BPE 4k', 'BPE 8k', 'Unigram 4k', 'Unigram 8k', 'PRPE\nmorfológico']
fertility_vals = [2.2331, 1.9177, 2.3178, 1.9533, 1.7971]
colores = ['#E24B4A','#E24B4A','#1D9E75','#1D9E75','#378ADD']
alphas = [0.5, 1.0, 0.5, 1.0, 1.0]

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(modelos, fertility_vals, color=colores, 
              alpha=1.0, width=0.6, edgecolor='none')

# Resaltar PRPE
bars[4].set_linewidth(2)
bars[4].set_edgecolor('#1a5276')

for bar, val in zip(bars, fertility_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            str(val), ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylim(0, 2.8)
ax.set_ylabel('Fertility rate (tokens/palabra)', fontsize=11)
ax.set_title('QuechuaTok: Comparativa de tokenizadores — Quechua del Sur\n← menor fertility = mejor segmentación', fontsize=11)
ax.axhline(y=1.7971, color='#378ADD', linestyle='--', alpha=0.4, linewidth=1)
ax.set_facecolor('#fafafa')
fig.patch.set_facecolor('white')

plt.tight_layout()
plt.savefig('quechuatok_final_benchmark.png', dpi=200, bbox_inches='tight')
plt.show()
print('Final chart saved ✓')

In [ ]:
# Expand corpus with additional sources
from datasets import load_dataset

# somosnlp - lado quechua
ds_somo = load_dataset("somosnlp-hackathon-2022/spanish-to-quechua", split="train")
somo_texts = [item.get("target", item.get("quechua", "")) 
              for item in ds_somo 
              if item.get("target", item.get("quechua", "")).strip()]
print(f"somosnlp: {len(somo_texts):,} sentences")

# Combine with original corpus
all_texts = train_sents + somo_texts
all_texts = list(dict.fromkeys(all_texts))  # deduplicate

# Guardar corpus expandido
with open('data/processed/quechua_expanded.txt', 'w') as f:
    f.write('\n'.join(all_texts))

print(f"Corpus expandido: {len(all_texts):,} total lines")

In [ ]:
somo_texts = [item['qu'] for item in ds_somo 
              if item.get('qu', '').strip()]
print(f"somosnlp: {len(somo_texts):,} sentences")

# Combine with original corpus
all_texts = train_sents + somo_texts
all_texts = list(dict.fromkeys(all_texts))  # deduplicate

with open('data/processed/quechua_expanded.txt', 'w') as f:
    f.write('\n'.join(all_texts))

print(f"Corpus expandido: {len(all_texts):,} total lines")

In [ ]:
VOCAB3 = 16000
TRAIN_EXP = 'data/processed/quechua_expanded.txt'

# BPE 16k
spm.SentencePieceTrainer.train(
    input=TRAIN_EXP,
    model_prefix='models/bpe_16k',
    vocab_size=VOCAB3,
    model_type='bpe',
    character_coverage=0.9999,
    byte_fallback=True,
    user_defined_symbols=["'"]
)
bpe_16k = spm.SentencePieceProcessor()
bpe_16k.load('models/bpe_16k.model')
print('BPE 16k ✓')

# Unigram 16k
spm.SentencePieceTrainer.train(
    input=TRAIN_EXP,
    model_prefix='models/unigram_16k',
    vocab_size=VOCAB3,
    model_type='unigram',
    character_coverage=0.9999,
    byte_fallback=True,
    user_defined_symbols=["'"]
)
uni_16k = spm.SentencePieceProcessor()
uni_16k.load('models/unigram_16k.model')
print('Unigram 16k ✓')

In [ ]:
print(f"{'Palabra':<25} {'Gold':<28} {'BPE 16k':<35} {'Unigram 16k'}")
print('─' * 110)
for word, gold in examples2:
    bpe_seg  = ' | '.join(bpe_16k.encode(word, out_type=str))
    uni_seg  = ' | '.join(uni_16k.encode(word, out_type=str))
    print(f"{word:<25} {gold:<28} {bpe_seg:<35} {uni_seg}")

In [ ]:
import matplotlib.pyplot as plt

modelos = ['BPE\n4k', 'BPE\n8k', 'BPE\n16k', 
           'Unigram\n4k', 'Unigram\n8k', 'Unigram\n16k',
           'PRPE\nmorphological']
fertility_vals = [2.2331, 1.9177, 1.636, 2.3178, 1.9533, 1.6378, 1.7971]
colores = ['#E24B4A','#E24B4A','#E24B4A',
           '#1D9E75','#1D9E75','#1D9E75','#378ADD']

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(modelos, fertility_vals, color=colores, width=0.6, edgecolor='none')

ax.annotate('Lower fertility\nbut incorrect\nmorphology ✗',
            xy=(2, 1.636), xytext=(2, 2.1),
            arrowprops=dict(arrowstyle='->', color='#E24B4A'),
            ha='center', fontsize=9, color='#E24B4A')

ax.annotate('Lower fertility\nAND correct\nmorphology ✓',
            xy=(6, 1.7971), xytext=(5.5, 2.2),
            arrowprops=dict(arrowstyle='->', color='#378ADD'),
            ha='center', fontsize=9, color='#378ADD')

for bar, val in zip(bars, fertility_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            str(val), ha='center', va='bottom', fontsize=9, fontweight='bold')

ax.set_ylim(0, 2.9)
ax.set_ylabel('Fertility rate (tokens/word)', fontsize=11)
ax.set_title('QuechuaTok: Fertility Rate is Insufficient to Evaluate\n'
             'Tokenizers in Agglutinative Languages', fontsize=11)
ax.set_facecolor('#fafafa')
fig.patch.set_facecolor('white')

plt.tight_layout()
plt.savefig('quechuatok_figure1.png', dpi=300, bbox_inches='tight')
plt.show()
print('Saved ✓')

## 6. Final Benchmark — Complete Results

Complete comparison across all 7 tokenizers with 3 metrics.
See paper: [arXiv coming soon]

In [ ]:
# Tabla final completa — todos los modelos + 3 métricas
import pandas as pd

final_results = {
    'BPE 4k':      {'Fertility↓': 2.2331, 'OOV%↓': 0.0, 'MorphAcc%↑': 6.67},
    'BPE 8k':      {'Fertility↓': 1.9177, 'OOV%↓': 0.0, 'MorphAcc%↑': 6.67},
    'BPE 16k':     {'Fertility↓': 1.6360, 'OOV%↓': 0.0, 'MorphAcc%↑': 6.67},
    'Unigram 4k':  {'Fertility↓': 2.3178, 'OOV%↓': 0.0, 'MorphAcc%↑': 66.67},
    'Unigram 8k':  {'Fertility↓': 1.9533, 'OOV%↓': 0.0, 'MorphAcc%↑': 26.67},
    'Unigram 16k': {'Fertility↓': 1.6378, 'OOV%↓': 0.0, 'MorphAcc%↑': 33.33},
    'PRPE':        {'Fertility↓': 1.7971, 'OOV%↓': 0.0, 'MorphAcc%↑': 83.33},
}

df = pd.DataFrame(final_results).T
print("=== QuechuaTok Final Benchmark ===")
print(df.to_string())
print("\n↓ lower is better  ↑ higher is better")
print("\nKey finding: BPE achieves lowest fertility by memorization (6.67% MorphAcc)")
print("PRPE achieves highest morphological accuracy (83.33%) by design")

## 7. Morphological Silver Standard — SQUOIA Analyzer (Rios, 2016)

Morphological boundary accuracy is evaluated against the SQUOIA finite-state 
morphological analyzer. This section installs Foma and compiles the analyzer 
for full reproducibility.

Reference: Rios, A. (2016). A Basic Language Technology Toolkit for Quechua.
Repository: https://github.com/ariosquoia/squoia

In [ ]:
import subprocess

# Install Foma finite-state compiler
result = subprocess.run(['apt-get', 'install', '-y', 'foma-bin'],
                       capture_output=True, text=True)
print("Foma installed ✓" if result.returncode == 0 else result.stderr)

# Clone SQUOIA repository
result2 = subprocess.run(['git', 'clone', 
                          'https://github.com/ariosquoia/squoia.git'],
                         capture_output=True, text=True)
print("SQUOIA cloned ✓" if result2.returncode == 0 else result2.stderr)

In [ ]:
import os

os.chdir('squoia/morphology/spellChecker/analyzer')

result = subprocess.run(['foma', '-e', 'source analyzeUnificado.foma',
                         '-e', 'save stack analyzeUnificado.bin',
                         '-e', 'quit'],
                        capture_output=True, text=True, timeout=120)

os.chdir('/kaggle/working')
print("Analyzer compiled ✓" if result.returncode == 0 else result.stderr[-300:])

In [ ]:
def analyze_quechua(words: list[str]) -> dict:
    """
    Analyzes Quechua words using the SQUOIA morphological analyzer (Rios, 2016).
    Returns dict {word: [morphological_analyses]}
    Silver standard for MorphAcc evaluation.
    """
    result = subprocess.run(
        ['flookup', 'squoia/morphology/spellChecker/analyzer/analyzeUnificado.bin'],
        input='\n'.join(words),
        capture_output=True, text=True, timeout=30
    )
    analyses = {}
    for line in result.stdout.strip().split('\n'):
        if '\t' in line:
            word, analysis = line.split('\t', 1)
            if word not in analyses:
                analyses[word] = []
            if analysis != '+?':
                analyses[word].append(analysis)
    return analyses

# Verify analyzer works
test = analyze_quechua(['rimani', 'wasipi', 'mikhuni', 'allinmi'])
print(f"{'Word':<12} {'SQUOIA Analysis'}")
print('─' * 40)
for word, ana in test.items():
    print(f"{word:<12} {ana[0] if ana else '+? (unknown)'}")
print("\nSQUOIA analyzer ready ✓")

In [ ]:
# Silver standard dataset — verified against SQUOIA output
silver_dataset = {
    'rimani':    ('=hablar+1.Sg.Subj',         ['rima', 'ni']),
    'rimanki':   ('=hablar+2.Sg.Subj',         ['rima', 'nki']),
    'riman':     ('=hablar+3.Sg.Subj',         ['rima', 'n']),
    'rimanchik': ('=hablar+1.Pl.Incl.Subj',    ['rima', 'nchik']),
    'rimanku':   ('=hablar+3.Pl.Subj',         ['rima', 'nku']),
    'apani':     ('=llevar+1.Sg.Subj',         ['apa', 'ni']),
    'apanki':    ('=llevar+2.Sg.Subj',         ['apa', 'nki']),
    'apan':      ('=llevar+3.Sg.Subj',         ['apa', 'n']),
    'purin':     ('=andar+3.Sg.Subj',          ['puri', 'n']),
    'purinki':   ('=andar+2.Sg.Subj',          ['puri', 'nki']),
    'wasipi':    ('=casa+Loc',                 ['wasi', 'pi']),
    'mikhuni':   ('=comer+1.Sg.Subj',          ['mikhu', 'ni']),
    'mikhuyki':  ('=comer+1.Sg.Subj_2.Sg.Obj', ['mikhu', 'yki']),
    'allin':     ('=bien,bueno',               ['allin']),
    'allinmi':   ('=bien,bueno+DirE',          ['allin', 'mi']),
}

def morpheme_boundary_accuracy(pred_segs, silver_morphemes):
    pred_set = set(pred_segs)
    silver_set = set(silver_morphemes)
    if not silver_set:
        return 0.0
    return round(len(pred_set & silver_set) / len(silver_set), 4)

# Evaluate all tokenizers
print(f"{'Model':<16} {'MorphAcc%'}")
print('─' * 30)
for name, tok, typ in [
    ('BPE 4k',      bpe,     'spm'),
    ('BPE 8k',      bpe_8k,  'spm'),
    ('BPE 16k',     bpe_16k, 'spm'),
    ('Unigram 4k',  uni,     'spm'),
    ('Unigram 8k',  uni_8k,  'spm'),
    ('Unigram 16k', uni_16k, 'spm'),
]:
    scores = []
    for word, (_, silver) in silver_dataset.items():
        seg = [s.replace('▁','') for s in tok.encode(word, out_type=str)]
        scores.append(morpheme_boundary_accuracy(seg, silver))
    print(f"{name:<16} {round(sum(scores)/len(scores)*100, 2)}")

# PRPE
prpe_scores = []
for word, (_, silver) in silver_dataset.items():
    prpe_scores.append(morpheme_boundary_accuracy(segment_prpe(word), silver))
print(f"{'PRPE':<16} {round(sum(prpe_scores)/len(prpe_scores)*100, 2)}")
print("\nSilver standard: SQUOIA analyzer (Rios, 2016)")

In [ ]:
import sentencepiece as spm
import numpy as np, math, os, re, unicodedata
from collections import Counter
from pathlib import Path
from datasets import load_dataset

Path('models').mkdir(exist_ok=True)
Path('data/processed').mkdir(parents=True, exist_ok=True)

# 1. Corpus
print("Downloading...")
ds1 = load_dataset('Llamacha/monolingual-quechua-iic', split='train')
texts = [item['text'] for item in ds1 if item.get('text','').strip()]
TRAIN = 'data/processed/corpus.txt'
with open(TRAIN, 'w') as f:
    f.write('\n'.join(texts[:50000]))
print(f"Corpus: {min(len(texts),50000)} lines ✓")

# 2. Train tokenizers
for vocab, name, mtype in [
    (4000,'bpe','bpe'), (16000,'bpe_16k','bpe'),
    (4000,'unigram','unigram'), (16000,'unigram_16k','unigram')]:
    spm.SentencePieceTrainer.train(
        input=TRAIN, model_prefix=f'models/{name}',
        vocab_size=vocab, model_type=mtype,
        character_coverage=0.9999, byte_fallback=True,
        user_defined_symbols=["'"])
    print(f'{name} ✓')

bpe = spm.SentencePieceProcessor(); bpe.load('models/bpe.model')
uni = spm.SentencePieceProcessor(); uni.load('models/unigram.model')
bpe_16k = spm.SentencePieceProcessor(); bpe_16k.load('models/bpe_16k.model')
uni_16k = spm.SentencePieceProcessor(); uni_16k.load('models/unigram_16k.model')
print("All models loaded ✓")

# 3. PRPE
SUFFIXES_PRPE = sorted([
    'ni','nki','n','nchik','nkichik','nku',
    'rqa','sqa','nqa','rqan','sqan',
    'kuna','pi','manta','wan','paq','ta','qa','pas',
    'mi','si','cha','stin','spa','na',
], key=len, reverse=True)

def segment_prpe(word):
    if len(word) <= 3: return [word]
    segments, remaining = [], word
    while len(remaining) > 3:
        found = False
        for suf in SUFFIXES_PRPE:
            if remaining.endswith(suf) and len(remaining)-len(suf) >= 3:
                segments.insert(0, suf)
                remaining = remaining[:-len(suf)]
                found = True
                break
        if not found: break
    segments.insert(0, remaining)
    return segments

# 4. Bigram perplexity evaluation
test_lines = texts[50000:51000] if len(texts) > 51000 else texts[-1000:]

def compute_bigram_perplexity(sentences, tokenize_fn):
    tokenized = [tokenize_fn(s) for s in sentences]
    split = int(len(tokenized) * 0.8)
    train_tok, eval_tok = tokenized[:split], tokenized[split:]
    
    bigram_counts, unigram_counts = Counter(), Counter()
    for tokens in train_tok:
        tokens = ['<s>'] + tokens + ['</s>']
        for i in range(len(tokens)-1):
            bigram_counts[(tokens[i], tokens[i+1])] += 1
            unigram_counts[tokens[i]] += 1
        unigram_counts['</s>'] += 1
    
    vocab_size = len(unigram_counts)
    log_prob, n_tokens = 0, 0
    for tokens in eval_tok:
        tokens = ['<s>'] + tokens + ['</s>']
        for i in range(len(tokens)-1):
            bg = bigram_counts.get((tokens[i], tokens[i+1]), 0)
            ug = unigram_counts.get(tokens[i], 0)
            log_prob += math.log2((bg+1)/(ug+vocab_size))
            n_tokens += 1
    
    return round(2**(-log_prob/n_tokens), 2), vocab_size

print(f"\n{'Model':<16} {'Perplexity↓':<14} {'Vocab'}")
print('─' * 44)
for name, fn in [
    ('BPE 4k',     lambda s: bpe.encode(s, out_type=str)),
    ('BPE 16k',    lambda s: bpe_16k.encode(s, out_type=str)),
    ('Unigram 4k', lambda s: uni.encode(s, out_type=str)),
    ('Unigram 16k',lambda s: uni_16k.encode(s, out_type=str)),
    ('PRPE',       lambda s: [seg for w in s.split() for seg in segment_prpe(w)]),
]:
    ppl, v = compute_bigram_perplexity(test_lines, fn)
    print(f"{name:<16} {ppl:<14} {v}")

In [ ]:
print("=== QuechuaTok: Complete Results ===\n")
print(f"{'Model':<16} {'Fertility↓':<12} {'MorphAcc%↑':<14} {'Perplexity↓'}")
print('─' * 56)
data = [
    ('BPE 4k',      2.2331, 6.67,  1556.02),
    ('BPE 16k',     1.6360, 6.67,  2553.05),
    ('Unigram 4k',  2.3178, 66.67, 1344.34),
    ('Unigram 16k', 1.6378, 33.33, 2091.81),
    ('PRPE',        1.7971, 83.33, 1879.36),
]
for name, fert, morph, ppl in data:
    print(f"{name:<16} {fert:<12} {morph:<14} {ppl}")

print("\n↓ lower is better  ↑ higher is better")
print("\nKey findings:")
print("1. PRPE: highest MorphAcc (83.33%) — best morphological quality")
print("2. Unigram 4k: lowest perplexity (1344) — best sequence predictability")
print("3. BPE: lowest fertility but worst MorphAcc AND high perplexity")
print("4. No single tokenizer wins all metrics — metric choice matters")

## Citation

If you use this work, please cite:

```bibtex
@misc{contreras2026quechuatok,
  title={QuechuaTok: Morphological Boundary Accuracy as a Necessary Metric for Tokenizer Evaluation in Agglutinative Low-Resource Languages},
  author={Contreras, Maria},
  year={2026},
  eprint={2606.23943},
  archivePrefix={arXiv},
  primaryClass={cs.CL},
  url={https://arxiv.org/abs/2606.23943}
}
```

**Links:** [arXiv](https://arxiv.org/abs/2606.23943) | [GitHub](https://github.com/mcontrerasmalpar-pixel/QuechuaTok) | [Kaggle](https://kaggle.com/macmaky)